In [ ]:
# ============================================================================
# QUANTUM-ENHANCED LABS SOLVER - CUDA-Q IMPLEMENTATION
# Complete End-to-End Implementation
# ============================================================================
# This code implements a hybrid quantum-classical optimizer for the
# Low Autocorrelation Binary Sequences (LABS) problem using:
# - Counteradiabatic quantum circuit (CUDA-Q) for initial population generation
# - Memetic Tabu Search (MTS) for classical refinement
# ============================================================================

import cudaq
import numpy as np
import random
from typing import List

# Set the target (use 'nvidia' for GPU, 'qpp-cpu' for CPU simulation)
# cudaq.set_target('nvidia')  # Uncomment for GPU acceleration
cudaq.set_target('qpp-cpu')   # CPU simulator

# ============================================================================
# PART 1: LABS PROBLEM UTILITIES
# ============================================================================

def labs_energy(s):
    """
    Compute LABS energy for sequence s in {-1, +1}^N.
    E = sum_{k=1}^{N-1} C_k^2 where C_k = sum_{i} s_i * s_{i+k}
    
    Lower energy = better radar/communication sequence.
    """
    N = len(s)
    energy = 0
    for k in range(1, N):
        Ck = sum(s[idx] * s[idx + k] for idx in range(N - k))
        energy += Ck * Ck
    return energy


def bitstring_to_spin(bitstring):
    """Convert '01...' bitstring to +1/-1 spin array."""
    return np.array([1 if b == '1' else -1 for b in bitstring])


def spin_to_bitstring(spin):
    """Convert +1/-1 spin array to '01...' bitstring."""
    return ''.join(['1' if s == 1 else '0' for s in spin])


def get_interactions(N):
    """
    Generate 2-body (G2) and 4-body (G4) interaction indices.
    These define which qubits interact in the LABS Hamiltonian.
    Based on Eq. 15 from the counteradiabatic LABS paper.
    """
    G2 = []
    G4 = []
    
    # Two-body terms from autocorrelation structure
    for i in range(N - 1):
        for k in range(1, N - i):
            G2.append([i, i + k])
    
    # Four-body terms from cross-correlation products
    for i in range(N - 3):
        for t in range(1, (N - i) // 2 + 1):
            for k in range(t + 1, N - i - t + 1):
                final_idx = i + k + t
                if final_idx < N:
                    G4.append([i, i + t, i + k, final_idx])
    
    return G2, G4


# Known optimal LABS energies for validation
KNOWN_OPTIMAL_ENERGIES = {
    3: 1,
    4: 2,
    5: 4,
    6: 5,
    7: 9,
    8: 10,
    9: 12,
    10: 13,
    11: 17,
    12: 20,
    13: 24,
    14: 28,
    15: 32,
    20: 56,
}





In [ ]:
# ============================================================================
# PART 2: CUDA-Q QUANTUM CIRCUIT BUILDING BLOCKS
# ============================================================================

@cudaq.kernel
def rzz_gate(q0: cudaq.qubit, q1: cudaq.qubit, theta: float):
    """RZZ(theta) = exp(-i * theta/2 * Z tensor Z) gate."""
    x.ctrl(q0, q1)
    rz(theta, q1)
    x.ctrl(q0, q1)


@cudaq.kernel
def two_qubit_block(q0: cudaq.qubit, q1: cudaq.qubit, theta: float):
    """
    Two-qubit counteradiabatic block implementing RYZ and RZY terms.
    Creates correlated rotations between qubit pairs.
    """
    # RYZ term: exp(-i * theta * Y_0 tensor Z_1)
    rx(1.5707963267948966, q0)  # pi/2
    x.ctrl(q0, q1, q2)
    x.ctrl(q0, q1)
    x.ctrl(q1, q2)
    x.ctrl(q2, q3)
    rz(theta, q3)
    x.ctrl(q2, q3)
    x.ctrl(q1, q2)
    x.ctrl(q0, q1)
    ry(-1.5707963267948966, q2)
    
    # RZZZY block
    ry(1.5707963267948966, q3)
    x.ctrl(q0, q1)
    x.ctrl(q1, q2)
    x.ctrl(q2, q3)
    rz(theta, q3)
    x.ctrl(q2, q3)
    x.ctrl(q1, q2)
    x.ctrl(q0, q1)
    ry(-1.5707963267948966, q3)



In [ ]:
# ============================================================================
# PART 3: COUNTERADIABATIC CIRCUIT KERNELS FOR DIFFERENT N
# ============================================================================

def compute_theta(t: float, dt: float, T: float, N: int) -> float:
    """
    Compute rotation angle using sinusoidal annealing schedule.
    Based on counteradiabatic driving theory.
    """
    lam = np.sin(np.pi * t / (2 * T)) ** 2
    dlam = (np.pi / (2 * T)) * np.sin(np.pi * t / T)
    return dlam * dt / (N * (1 - lam + 0.1))


# --- N=3 Circuit ---
@cudaq.kernel
def labs_circuit_n3(theta: float):
    qubits = cudaq.qvector(3)
    
    # Initial superposition
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    
    # Two-body terms for N=3: G2 = [[0,1], [0,2], [1,2]]
    two_qubit_block(qubits[0], qubits[1], theta)
    two_qubit_block(qubits[0], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[2], theta)
    
    # Measurement
    mz(qubits)


# --- N=4 Circuit ---
@cudaq.kernel
def labs_circuit_n4(theta: float):
    qubits = cudaq.qvector(4)
    
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[3])
    
    # Two-body terms
    two_qubit_block(qubits[0], qubits[1], theta)
    two_qubit_block(qubits[0], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[3], theta)
    
    # Four-body term: [0, 1, 2, 3]
    four_qubit_block(qubits[0], qubits[1], qubits[2], qubits[3], theta * 0.5)
    
    mz(qubits)


# --- N=5 Circuit ---
@cudaq.kernel
def labs_circuit_n5(theta: float):
    qubits = cudaq.qvector(5)
    
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[3])
    h(qubits[4])
    
    # Two-body terms
    two_qubit_block(qubits[0], qubits[1], theta)
    two_qubit_block(qubits[0], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[4], theta)
    two_qubit_block(qubits[3], qubits[4], theta)
    
    # Four-body terms
    four_qubit_block(qubits[0], qubits[1], qubits[2], qubits[3], theta * 0.5)
    four_qubit_block(qubits[1], qubits[2], qubits[3], qubits[4], theta * 0.5)
    
    mz(qubits)


# --- N=6 Circuit ---
@cudaq.kernel
def labs_circuit_n6(theta: float):
    qubits = cudaq.qvector(6)
    
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[3])
    h(qubits[4])
    h(qubits[5])
    
    # Two-body terms
    two_qubit_block(qubits[0], qubits[1], theta)
    two_qubit_block(qubits[0], qubits[2], theta)
    two_qubit_block(qubits[0], qubits[3], theta)
    two_qubit_block(qubits[1], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[3], theta)
    two_qubit_block(qubits[1], qubits[4], theta)
    two_qubit_block(qubits[2], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[4], theta)
    two_qubit_block(qubits[2], qubits[5], theta)
    two_qubit_block(qubits[3], qubits[4], theta)
    two_qubit_block(qubits[3], qubits[5], theta)
    two_qubit_block(qubits[4], qubits[5], theta)
    
    # Four-body terms
    four_qubit_block(qubits[0], qubits[1], qubits[2], qubits[3], theta * 0.5)
    four_qubit_block(qubits[0], qubits[1], qubits[3], qubits[4], theta * 0.5)
    four_qubit_block(qubits[1], qubits[2], qubits[3], qubits[4], theta * 0.5)
    four_qubit_block(qubits[1], qubits[2], qubits[4], qubits[5], theta * 0.5)
    four_qubit_block(qubits[2], qubits[3], qubits[4], qubits[5], theta * 0.5)
    
    mz(qubits)


# --- N=7 Circuit ---
@cudaq.kernel
def labs_circuit_n7(theta: float):
    qubits = cudaq.qvector(7)
    
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[3])
    h(qubits[4])
    h(qubits[5])
    h(qubits[6])
    
    # Two-body terms
    two_qubit_block(qubits[0], qubits[1], theta)
    two_qubit_block(qubits[0], qubits[2], theta)
    two_qubit_block(qubits[0], qubits[3], theta)
    two_qubit_block(qubits[1], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[3], theta)
    two_qubit_block(qubits[1], qubits[4], theta)
    two_qubit_block(qubits[2], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[4], theta)
    two_qubit_block(qubits[2], qubits[5], theta)
    two_qubit_block(qubits[3], qubits[4], theta)
    two_qubit_block(qubits[3], qubits[5], theta)
    two_qubit_block(qubits[3], qubits[6], theta)
    two_qubit_block(qubits[4], qubits[5], theta)
    two_qubit_block(qubits[4], qubits[6], theta)
    two_qubit_block(qubits[5], qubits[6], theta)
    
    # Four-body terms
    four_qubit_block(qubits[0], qubits[1], qubits[2], qubits[3], theta * 0.5)
    four_qubit_block(qubits[0], qubits[1], qubits[3], qubits[4], theta * 0.5)
    four_qubit_block(qubits[1], qubits[2], qubits[3], qubits[4], theta * 0.5)
    four_qubit_block(qubits[1], qubits[2], qubits[4], qubits[5], theta * 0.5)
    four_qubit_block(qubits[2], qubits[3], qubits[4], qubits[5], theta * 0.5)
    four_qubit_block(qubits[2], qubits[3], qubits[5], qubits[6], theta * 0.5)
    four_qubit_block(qubits[3], qubits[4], qubits[5], qubits[6], theta * 0.5)
    
    mz(qubits)


# --- N=8 Circuit ---
@cudaq.kernel
def labs_circuit_n8(theta: float):
    qubits = cudaq.qvector(8)
    
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[3])
    h(qubits[4])
    h(qubits[5])
    h(qubits[6])
    h(qubits[7])
    
    # Two-body terms (selected subset for efficiency)
    two_qubit_block(qubits[0], qubits[1], theta)
    two_qubit_block(qubits[0], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[4], theta)
    two_qubit_block(qubits[3], qubits[4], theta)
    two_qubit_block(qubits[3], qubits[5], theta)
    two_qubit_block(qubits[4], qubits[5], theta)
    two_qubit_block(qubits[4], qubits[6], theta)
    two_qubit_block(qubits[5], qubits[6], theta)
    two_qubit_block(qubits[5], qubits[7], theta)
    two_qubit_block(qubits[6], qubits[7], theta)
    
    # Four-body terms (selected subset)
    four_qubit_block(qubits[0], qubits[1], qubits[2], qubits[3], theta * 0.5)
    four_qubit_block(qubits[1], qubits[2], qubits[3], qubits[4], theta * 0.5)
    four_qubit_block(qubits[2], qubits[3], qubits[4], qubits[5], theta * 0.5)
    four_qubit_block(qubits[3], qubits[4], qubits[5], qubits[6], theta * 0.5)
    four_qubit_block(qubits[4], qubits[5], qubits[6], qubits[7], theta * 0.5)
    
    mz(qubits)


# --- N=9 Circuit ---
@cudaq.kernel
def labs_circuit_n9(theta: float):
    qubits = cudaq.qvector(9)
    
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[3])
    h(qubits[4])
    h(qubits[5])
    h(qubits[6])
    h(qubits[7])
    h(qubits[8])
    
    # Two-body terms (chain connectivity)
    two_qubit_block(qubits[0], qubits[1], theta)
    two_qubit_block(qubits[1], qubits[2], theta)
    two_qubit_block(qubits[2], qubits[3], theta)
    two_qubit_block(qubits[3], qubits[4], theta)
    two_qubit_block(qubits[4], qubits[5], theta)
    two_qubit_block(qubits[5], qubits[6], theta)
    two_qubit_block(qubits[6], qubits[7], theta)
    two_qubit_block(qubits[7], qubits[8], theta)
    two_qubit_block(qubits[0], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[4], theta)
    two_qubit_block(qubits[3], qubits[5], theta)
    two_qubit_block(qubits[4], qubits[6], theta)
    two_qubit_block(qubits[5], qubits[7], theta)
    two_qubit_block(qubits[6], qubits[8], theta)
    
    # Four-body terms
    four_qubit_block(qubits[0], qubits[1], qubits[2], qubits[3], theta * 0.5)
    four_qubit_block(qubits[2], qubits[3], qubits[4], qubits[5], theta * 0.5)
    four_qubit_block(qubits[4], qubits[5], qubits[6], qubits[7], theta * 0.5)
    four_qubit_block(qubits[5], qubits[6], qubits[7], qubits[8], theta * 0.5)
    
    mz(qubits)


# --- N=10 Circuit ---
@cudaq.kernel
def labs_circuit_n10(theta: float):
    qubits = cudaq.qvector(10)
    
    h(qubits[0])
    h(qubits[1])
    h(qubits[2])
    h(qubits[3])
    h(qubits[4])
    h(qubits[5])
    h(qubits[6])
    h(qubits[7])
    h(qubits[8])
    h(qubits[9])
    
    # Two-body terms
    two_qubit_block(qubits[0], qubits[1], theta)
    two_qubit_block(qubits[1], qubits[2], theta)
    two_qubit_block(qubits[2], qubits[3], theta)
    two_qubit_block(qubits[3], qubits[4], theta)
    two_qubit_block(qubits[4], qubits[5], theta)
    two_qubit_block(qubits[5], qubits[6], theta)
    two_qubit_block(qubits[6], qubits[7], theta)
    two_qubit_block(qubits[7], qubits[8], theta)
    two_qubit_block(qubits[8], qubits[9], theta)
    two_qubit_block(qubits[0], qubits[2], theta)
    two_qubit_block(qubits[1], qubits[3], theta)
    two_qubit_block(qubits[2], qubits[4], theta)
    two_qubit_block(qubits[3], qubits[5], theta)
    two_qubit_block(qubits[4], qubits[6], theta)
    two_qubit_block(qubits[5], qubits[7], theta)
    two_qubit_block(qubits[6], qubits[8], theta)
    two_qubit_block(qubits[7], qubits[9], theta)
    
    # Four-body terms
    four_qubit_block(qubits[0], qubits[1], qubits[2], qubits[3], theta * 0.5)
    four_qubit_block(qubits[2], qubits[3], qubits[4], qubits[5], theta * 0.5)
    four_qubit_block(qubits[4], qubits[5], qubits[6], qubits[7], theta * 0.5)
    four_qubit_block(qubits[6], qubits[7], qubits[8], qubits[9], theta * 0.5)
    
    mz(qubits)





In [ ]:
# ============================================================================
# PART 4: CIRCUIT SELECTOR
# ============================================================================

def get_labs_circuit(N: int):
    """Return the appropriate LABS circuit kernel for given N."""
    circuits = {
        3: labs_circuit_n3,
        4: labs_circuit_n4,
        5: labs_circuit_n5,
        6: labs_circuit_n6,
        7: labs_circuit_n7,
        8: labs_circuit_n8,
        9: labs_circuit_n9,
        10: labs_circuit_n10,
    }
    if N not in circuits:
        raise ValueError(f"N={N} not supported. Use N in [3, 10].")
    return circuits[N]


In [ ]:

# ============================================================================
# PART 5: CLASSICAL MEMETIC TABU SEARCH
# ============================================================================

def combine(p1, p2):
    """Single-point crossover of two parent sequences."""
    N = len(p1)
    k = random.randint(1, N - 1)
    return np.concatenate([p1[:k], p2[k:]])


def mutate(s, p_mut=0.05):
    """Flip each spin with probability p_mut."""
    s_new = s.copy()
    for i in range(len(s)):
        if random.random() < p_mut:
            s_new[i] *= -1
    return s_new


def tabu_search(s_init, tabu_size=20, max_iters=50):
    """
    Local search with tabu list to avoid cycling.
    Explores neighbors by single bit flips.
    """
    s = s_init.copy()
    best_s = s.copy()
    best_energy = labs_energy(s)
    tabu_list = []
    
    for _ in range(max_iters):
        neighbors = []
        for i in range(len(s)):
            neighbor = s.copy()
            neighbor[i] *= -1
            key = tuple(neighbor)
            if key not in tabu_list:
                neighbors.append((labs_energy(neighbor), neighbor))
        
        if not neighbors:
            break
        
        energy, s = min(neighbors, key=lambda x: x[0])
        tabu_list.append(tuple(s))
        if len(tabu_list) > tabu_size:
            tabu_list.pop(0)
        
        if energy < best_energy:
            best_energy = energy
            best_s = s.copy()
    
    return best_s, best_energy


def run_mts(N, pop_size=10, generations=10, p_mut=0.05, initial_population=None,
            verbose=False):
    """
    Full Memetic Tabu Search procedure.
    Can be seeded with quantum-sampled initial population.
    """
    # Initialize population
    if initial_population is not None:
        population = [np.array(s) for s in initial_population[:pop_size]]
        while len(population) < pop_size:
            population.append(2 * np.random.randint(0, 2, N) - 1)
    else:
        population = [2 * np.random.randint(0, 2, N) - 1 for _ in range(pop_size)]
    
    best_s = None
    best_energy = float('inf')
    
    for gen in range(generations):
        new_pop = []
        for _ in range(pop_size):
            p1, p2 = random.sample(population, 2)
            child = mutate(combine(p1, p2), p_mut)
            refined, energy = tabu_search(child)
            new_pop.append(refined)
            
            if energy < best_energy:
                best_energy = energy
                best_s = refined.copy()
                if verbose:
                    print(f"  Gen {gen}: New best energy = {best_energy}")
        
        population = new_pop
    
    final_energies = [labs_energy(s) for s in population]
    return best_s, best_energy, final_energies




In [ ]:

# ============================================================================
# PART 6: QUANTUM SAMPLING FUNCTION (CUDA-Q)
# ============================================================================

def sample_quantum_population(N, n_steps=2, T=1.0, shots=1000):
    """
    Sample bitstrings from the counteradiabatic circuit using CUDA-Q.
    
    Args:
        N: Number of qubits
        n_steps: Trotter steps (applied via multiple theta values)
        T: Evolution time
        shots: Number of samples
    
    Returns:
        List of spin arrays sampled from the quantum circuit
    """
    # Compute theta for final Trotter step
    dt = T / n_steps
    t = n_steps * dt
    theta = compute_theta(t, dt, T, N)
    
    # Get the appropriate circuit
    circuit = get_labs_circuit(N)
    
    # Sample using CUDA-Q
    result = cudaq.sample(circuit, theta, shots_count=shots)
    
    # Convert to spin arrays
    quantum_population = []
    for bitstring, count in result.items():
        # CUDA-Q returns bitstrings - convert to spin
        spin = bitstring_to_spin(bitstring)
        for _ in range(count):
            quantum_population.append(spin)
    
    return quantum_population, result


# ============================================================================
# PART 7: COMPLETE QUANTUM-ENHANCED WORKFLOW
# ============================================================================

def run_quantum_enhanced_labs(N, pop_size=20, generations=10, n_steps=2,
                               shots=1000, seed=42, verbose=True):
    """
    Complete quantum-enhanced LABS optimization workflow using CUDA-Q.
    """
    if verbose:
        print("=" * 60)
        print(f"QUANTUM-ENHANCED LABS SOLVER (CUDA-Q) - N={N}")
        print("=" * 60)
        
        optimal = KNOWN_OPTIMAL_ENERGIES.get(N, "unknown")
        print(f"Known optimal energy: {optimal}")
        print()
    
    # Set seeds
    np.random.seed(seed)
    random.seed(seed)
    
    # --- PHASE 1: Generate quantum population ---
    if verbose:
        print("Phase 1: Generating quantum-seeded population (CUDA-Q)...")
    
    quantum_pop, counts = sample_quantum_population(N, n_steps, shots=shots)
    
    # Analyze quantum samples
    quantum_energies = [labs_energy(s) for s in quantum_pop[:100]]
    if verbose:
        print(f"  Sampled {len(counts)} unique bitstrings")
        print(f"  Quantum sample avg energy: {np.mean(quantum_energies):.2f}")
        print(f"  Quantum sample min energy: {min(quantum_energies)}")
    
    # --- PHASE 2: Run classical MTS with random population ---
    if verbose:
        print("\nPhase 2a: Running Classical MTS (random start)...")
    
    np.random.seed(seed + 100)
    random.seed(seed + 100)
    
    best_random, energy_random, dist_random = run_mts(
        N=N,
        pop_size=pop_size,
        generations=generations,
        initial_population=None,
        verbose=verbose
    )
    
    # --- PHASE 3: Run MTS with quantum-seeded population ---
    if verbose:
        print("\nPhase 2b: Running Quantum-Enhanced MTS...")
    
    np.random.seed(seed + 200)
    random.seed(seed + 200)
    
    best_quantum, energy_quantum, dist_quantum = run_mts(
        N=N,
        pop_size=pop_size,
        generations=generations,
        initial_population=quantum_pop[:pop_size],
        verbose=verbose
    )
    
    # --- Results Summary ---
    if verbose:
        print("\n" + "=" * 60)
        print("RESULTS SUMMARY")
        print("=" * 60)
        print(f"Random MTS:   Best energy = {energy_random}")
        print(f"Quantum MTS:  Best energy = {energy_quantum}")
        
        optimal = KNOWN_OPTIMAL_ENERGIES.get(N)
        if optimal:
            print(f"Known optimal: {optimal}")
            if energy_quantum <= optimal:
                print(">>> QUANTUM MTS FOUND OPTIMAL SOLUTION!")
            elif energy_random <= optimal:
                print(">>> RANDOM MTS FOUND OPTIMAL SOLUTION!")
        
        improvement = energy_random - energy_quantum
        if improvement > 0:
            print(f"\nQuantum advantage: {improvement} energy units lower")
        elif improvement < 0:
            print(f"\nRandom was better by: {-improvement} energy units")
        else:
            print("\nBoth methods achieved same energy")
    
    return {
        "N": N,
        "best_random": best_random,
        "energy_random": energy_random,
        "dist_random": dist_random,
        "best_quantum": best_quantum,
        "energy_quantum": energy_quantum,
        "dist_quantum": dist_quantum,
        "quantum_samples": quantum_pop,
        "optimal": KNOWN_OPTIMAL_ENERGIES.get(N),
    }




In [ ]:
# ============================================================================
# PART 8: VALIDATION FUNCTIONS
# ============================================================================

def validate_symmetries(N=5):
    """Verify LABS energy respects known symmetries."""
    print(f"\nValidating LABS symmetries for N={N}...")
    
    s = 2 * np.random.randint(0, 2, N) - 1
    
    original_energy = labs_energy(s)
    inverted_energy = labs_energy(-s)
    reversed_energy = labs_energy(s[::-1])
    
    print(f"  Original:  {s} -> E = {original_energy}")
    print(f"  Inverted:  {-s} -> E = {inverted_energy}")
    print(f"  Reversed:  {s[::-1]} -> E = {reversed_energy}")
    
    assert original_energy == inverted_energy, "Inversion symmetry failed!"
    assert original_energy == reversed_energy, "Reversal symmetry failed!"
    print("  All symmetries verified!")
    
    return True


def validate_interactions(N=5):
    """Verify interaction indices are generated correctly."""
    print(f"\nValidating interaction indices for N={N}...")
    
    G2, G4 = get_interactions(N)
    
    print(f"  Two-body terms (G2): {len(G2)}")
    print(f"  Four-body terms (G4): {len(G4)}")
    
    for pair in G2:
        assert all(0 <= idx < N for idx in pair), f"G2 out of bounds: {pair}"
    
    for quartet in G4:
        assert all(0 <= idx < N for idx in quartet), f"G4 out of bounds: {quartet}"
    
    print("  All indices within bounds!")
    
    return True


def validate_known_optima():
    """Verify solver can find known optimal solutions for small N."""
    print("\n" + "=" * 60)
    print("VALIDATION: Testing against known optimal energies")
    print("=" * 60)
    
    results = []
    
    for N in [3, 4, 5, 6, 7]:
        optimal = KNOWN_OPTIMAL_ENERGIES[N]
        
        result = run_quantum_enhanced_labs(
            N=N,
            pop_size=15,
            generations=15,
            n_steps=2,
            shots=500,
            verbose=False
        )
        
        found = min(result["energy_random"], result["energy_quantum"])
        status = "PASS" if found <= optimal else "FAIL"
        
        results.append({
            "N": N,
            "optimal": optimal,
            "random": result["energy_random"],
            "quantum": result["energy_quantum"],
            "status": status
        })
        
        print(f"N={N}: Optimal={optimal}, Random={result['energy_random']}, "
              f"Quantum={result['energy_quantum']} [{status}]")
    
    return results



In [ ]:

# ============================================================================
# PART 9: VISUALIZATION (Optional)
# ============================================================================

def visualize_results(results):
    """Visualize comparison between random and quantum-seeded MTS."""
    try:
        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Plot 1: Energy distribution comparison
        ax1 = axes[0]
        ax1.hist(results["dist_random"], bins=15, alpha=0.5, 
                 label="Random MTS", color="blue")
        ax1.hist(results["dist_quantum"], bins=15, alpha=0.5, 
                 label="Quantum MTS", color="green")
        ax1.axvline(results["energy_random"], color="blue", 
                    linestyle="--", label=f"Random best: {results['energy_random']}")
        ax1.axvline(results["energy_quantum"], color="green", 
                    linestyle="--", label=f"Quantum best: {results['energy_quantum']}")
        if results["optimal"]:
            ax1.axvline(results["optimal"], color="red", 
                        linestyle="-", linewidth=2, label=f"Optimal: {results['optimal']}")
        ax1.set_xlabel("LABS Energy")
        ax1.set_ylabel("Frequency")
        ax1.set_title(f"Final Population Energy Distribution (N={results['N']})")
        ax1.legend()
        ax1.grid(alpha=0.3)
        
        # Plot 2: Best sequences
        ax2 = axes[1]
        width = 0.35
        x = np.arange(results["N"])
        ax2.bar(x - width/2, results["best_random"], width, 
                label=f"Random (E={results['energy_random']})", alpha=0.7)
        ax2.bar(x + width/2, results["best_quantum"], width, 
                label=f"Quantum (E={results['energy_quantum']})", alpha=0.7)
        ax2.set_xlabel("Position")
        ax2.set_ylabel("Spin Value")
        ax2.set_title("Best Sequences Found")
        ax2.set_xticks(x)
        ax2.legend()
        ax2.grid(alpha=0.3, axis="y")
        
        plt.tight_layout()
        plt.savefig(f"labs_cudaq_results_N{results['N']}.png", dpi=150)
        plt.show()
        
        print(f"\nFigure saved as 'labs_cudaq_results_N{results['N']}.png'")
        
    except ImportError:
        print("\nMatplotlib not available - skipping visualization")


# ============================================================================
# PART 10: MAIN EXECUTION
# ============================================================================

def main():
    """Main entry point for the CUDA-Q quantum-enhanced LABS solver."""
    
    print("=" * 70)
    print("QUANTUM-ENHANCED LABS SOLVER (CUDA-Q)")
    print("Low Autocorrelation Binary Sequences Optimization")
    print("=" * 70)
    print(f"CUDA-Q Target: {cudaq.get_target().name}")
    print()
    
    # --- Step 1: Run validations ---
    print("\n[STEP 1] Running Validations...")
    validate_symmetries(N=5)
    validate_interactions(N=5)
    
    # --- Step 2: Test against known optima ---
    print("\n[STEP 2] Testing against known optimal energies...")
    validation_results = validate_known_optima()
    
    # --- Step 3: Run full workflow ---
    print("\n[STEP 3] Running full quantum-enhanced workflow...")
    
    # Change N here to test different problem sizes (3-10)
    N = 7
    
    results = run_quantum_enhanced_labs(
        N=N,
        pop_size=20,
        generations=15,
        n_steps=2,
        shots=1000,
        seed=42,
        verbose=True
    )
    
    # --- Step 4: Visualize ---
    print("\n[STEP 4] Generating visualization...")
    visualize_results(results)
    
    # --- Step 5: Summary ---
    print("\n" + "=" * 70)
    print("EXECUTION COMPLETE")
    print("=" * 70)
    
    G2, G4 = get_interactions(N)
    print(f"\nCircuit Statistics for N={N}:")
    print(f"  Two-body terms: {len(G2)}")
    print(f"  Four-body terms: {len(G4)}")
    
    print("\nBest Results:")
    print(f"  Random MTS:  E = {results['energy_random']}")
    print(f"  Quantum MTS: E = {results['energy_quantum']}")
    if results['optimal']:
        print(f"  Known optimal: E = {results['optimal']}")
    
    return results


In [ ]:

# ============================================================================
# RUN THE CODE
# ============================================================================

if __name__ == "__main__":
    results = main()
